# Partie 1 - Analyse Exploratoire (EDA) - Retards des locations

**Objectif** : Répondre aux questions du Product Manager sur l'impact d'un délai minimum entre deux locations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option('display.max_columns', None)

In [ ]:
# 1. Chargement des données
data = pd.read_excel('../Data/get_around_delay_analysis.xlsx')

# Renommage des colonnes
data = data.rename(columns={
    "checkin_type": "type",
    "delay_at_checkout_in_minutes": "delay",
    "previous_ended_rental_id": "prev_id",
    "time_delta_with_previous_rental_in_minutes": "time_delta"
})

data.head()

In [ ]:
# 2. Statistiques et valeurs manquantes
data.info()

In [ ]:
# 3. Répartition des types de check-in
fig = px.pie(data, names='type', title="Répartition des types de check-in")
fig.show()

In [ ]:
# 4. Focus sur les locations terminées
data_enable = data.loc[data['state'] == 'ended', :].copy()
print(f"Locations terminées : {len(data_enable)}")

In [ ]:
# 5. Impact du seuil (threshold) sur les locations affectées
df_miss = pd.DataFrame(data_enable['time_delta'].unique(), columns=['threshold'])
df_miss = df_miss.dropna().sort_values('threshold')

data_mobile = data_enable.loc[data_enable['type'] == 'mobile']
data_connect = data_enable.loc[data_enable['type'] == 'connect']
nb_total = len(data_enable)

df_miss['mobile_data'] = df_miss['threshold'].apply(lambda thr: sum(data_mobile['time_delta'] < thr) * 100 / nb_total)
df_miss['connect_data'] = df_miss['threshold'].apply(lambda thr: sum(data_connect['time_delta'] < thr) * 100 / nb_total)
df_miss['global_data'] = df_miss['mobile_data'] + df_miss['connect_data']

fig = px.bar(df_miss, x='threshold', y=['mobile_data', 'connect_data', 'global_data'],
             barmode='group', labels={'value': 'Locations affectées (%)', 'threshold': 'Seuil (minutes)'})
fig.show()

In [ ]:
# 6. Proportion de retards
data_enable['late_or_early'] = data_enable['delay'].map(lambda v: 'En retard' if v > 0 else 'En avance')
fig = px.pie(data_enable, names='late_or_early', title="Retards vs avances")
fig.show()

In [ ]:
# 7. Cas problématiques (retard du précédent > time_delta)
data_join = data.merge(data[['rental_id', 'type', 'delay']],
                       how='inner', left_on='prev_id', right_on='rental_id',
                       suffixes=('_actuel', '_precedent'))

data_join['problematique'] = data_join['time_delta'] < data_join['delay_precedent']
data_join['problematique_label'] = data_join['problematique'].map(lambda v: 'Problématique' if v else 'Non problématique')

fig = px.pie(data_join, names='problematique_label', title="Cas problématiques")
fig.show()
print(f"% Cas problématiques : {data_join['problematique'].mean() * 100:.2f}%")

## 📊 Recommandations
- **Mobile** : seuil recommandé de **120 minutes**.
- **Connect** : seuil recommandé de **60 minutes**.